# BODAQS Batch Pre-processing Pipeline

Workflow:
1. Load CSVs
2. Canonicalise signal names
3. Normalize (zero + scale)
4. Estimate velocity/acceleration
5. Load event schema + detect events
6. Calculate event metrics
7. Write artifacts


In [1]:
import pandas as pd
import numpy as np
import logging
import importlib
import json
import os

from pathlib import Path

from bodaqs_analysis import run_macro
from bodaqs_analysis import normalize_and_scale
from bodaqs_analysis import estimate_va
from bodaqs_analysis import load_event_schema
from bodaqs_analysis import detect_events_from_schema
from bodaqs_analysis import extract_metrics_df
from bodaqs_analysis.pipeline import load_and_canonicalize
from bodaqs_analysis.bike_profile import load_bike_profile, resolve_normalization_ranges
import bodaqs_analysis.widgets.event_browser as eb
importlib.reload(eb)

from bodaqs_analysis.artifacts import (
    ArtifactStore,
    make_run_id,
    save_session_artifacts,
    write_run_manifest,
    write_session_manifest,
    write_events_partitioned_by_schema_id,
    write_metrics_partitioned_by_schema_id,
    copy_raw_csv_to_source,
    copy_session_aux_sources,
    ensure_run_is_new,
    ensure_session_is_new,
    update_manifest_description,
)

root = logging.getLogger()
root.setLevel(logging.DEBUG)

# Ensure any existing handlers (added by Jupyter) also filter at WARNING
for h in root.handlers:
    h.setLevel(logging.DEBUG)

# Optional: ensure BODAQS loggers don’t override root
logging.getLogger("bodaqs_analysis").setLevel(logging.NOTSET)  # inherit

fmt = logging.Formatter("%(levelname)s:%(name)s:%(message)s")
root = logging.getLogger()
root.setLevel(logging.INFO)

for h in root.handlers:
    h.setLevel(logging.INFO)
    h.setFormatter(fmt)


from IPython.display import display
from bodaqs_analysis.ui.preprocess_file_selector import PreprocessLogSelector

# --- Notebook parameters --------------------------------------------------
GENERIC_LOG_METADATA_PATH = Path("config/log_metadata_examples/current_logger_config_hr_timestamp_log_metadata2.json")
GENERIC_LOG_METADATA_PATHS = [GENERIC_LOG_METADATA_PATH] if GENERIC_LOG_METADATA_PATH is not None else None
BIKE_PROFILE_PATH = Path("config/bike_profiles/example_enduro_bike_v1.json")

selector = PreprocessLogSelector(artifacts_dir=Path("artifacts"))
display(selector.ui)


In [2]:
CSV_FILES = selector.get_selected_files()
if not CSV_FILES:
    raise ValueError(
        "No files are currently selected in the preprocess selector. "
        "Please select one or more rows in the grid, or use the selector's Select all visible button."
    )

logging.info("Found %d files:", len(CSV_FILES))
for p in CSV_FILES:
    logging.info("  %s", p.name)

# ---- Step 1: Load + canonicalise (no preprocess params yet) ----
sessions_step1 = {}
disp_cols_union = set()

for p in CSV_FILES:
    s = load_and_canonicalize(str(p), generic_log_metadata_paths=GENERIC_LOG_METADATA_PATHS)
    sid = str(s["session_id"])
    sessions_step1[sid] = s

    sig = (s.get("meta", {}) or {}).get("signals", {}) or {}
    disp_cols = [c for c, info in sig.items() if isinstance(info, dict) and info.get("quantity") == "disp"]
    disp_cols_union.update(disp_cols)

disp_cols_all = sorted(disp_cols_union)

logging.info("Displacement signals detected:")
for c in disp_cols_all:
    logging.info("  %s", c)

# Resolve bike-profile ranges once, for UI defaults and early validation.
BIKE_PROFILE = load_bike_profile(BIKE_PROFILE_PATH) if BIKE_PROFILE_PATH is not None else None
BIKE_PROFILE_NORMALIZE_RANGES = {}
if BIKE_PROFILE is not None:
    for s in sessions_step1.values():
        BIKE_PROFILE_NORMALIZE_RANGES.update(
            resolve_normalization_ranges(
                s,
                BIKE_PROFILE,
                bike_profile_path=BIKE_PROFILE_PATH,
                require_at_least_one=False,
            )
        )
    logging.info("Bike profile loaded: %s (%s)", BIKE_PROFILE.get("display_name"), BIKE_PROFILE_PATH)
    logging.info("Bike-profile normalization ranges resolved for %d signal(s)", len(BIKE_PROFILE_NORMALIZE_RANGES))

# (Optional) show any unclassified numeric signals so you can fix naming upstream
unclassified = []
for sid, s in sessions_step1.items():
    sig = (s.get("meta", {}) or {}).get("signals", {}) or {}
    for c, info in sig.items():
        if isinstance(info, dict) and info.get("quantity") is None:
            unclassified.append((sid, c, info.get("notes")))
if unclassified:
    print("\nUnclassified numeric columns (may need naming/normalization attention):")
    for sid, c, note in unclassified[:50]:
        print(f"  [{sid}] {c}  ({note})")

from bodaqs_analysis.pipeline import run_macro
from bodaqs_analysis.artifacts import (
    ArtifactStore, make_run_id, ensure_run_is_new, ensure_session_is_new,
    copy_raw_csv_to_source, copy_session_aux_sources, save_session_artifacts,
    write_session_manifest, write_run_manifest,
    write_events_partitioned_by_schema_id, write_metrics_partitioned_by_schema_id,
)

from bodaqs_analysis.ui.preprocess_controls import PreprocessControls, PreprocessDefaults
import ipywidgets as W
from IPython.display import display

# disp_cols_all and sessions_step1 come from Step 1
controls = PreprocessControls(
    disp_cols_all=disp_cols_all,
    sessions_by_id=sessions_step1,
    defaults=PreprocessDefaults(
        schema_path=r"event schema\event_schema - Basic.yaml",
        ingestion_mode="tolerant",
        prompt_for_descriptions=True,
        butterworth_generate_residuals=False,
    ),
    default_ranges=BIKE_PROFILE_NORMALIZE_RANGES,
)

display(controls.ui)

INFO:root:Found 1 files:
INFO:root:  2025-10-19_11-42-53_tombstone.CSV
INFO:bodaqs_analysis.io_logger:Logger same-stem sidecar not found: expected=C:\Users\Ben Connor\OneDrive\Documents\logs\2025-10-19_11-42-53_tombstone.json
INFO:bodaqs_analysis.io_logger:Logger generic sidecar search path(s): ['config\\sidecar_examples\\current_logger_config_hr_timestamp_sidecar2.json']
INFO:bodaqs_analysis.io_logger:Logger generic sidecar found: config\sidecar_examples\current_logger_config_hr_timestamp_sidecar2.json
INFO:bodaqs_analysis.io_logger:Logger sidecar loaded: path=config\sidecar_examples\current_logger_config_hr_timestamp_sidecar2.json sidecar_kind=generic selected_as_generic=True
INFO:bodaqs_analysis.io_logger:Logger sidecar optional column missing: sidecar_column_id=sample_id csv_ref={'by': 'header', 'header': 'sample_id'}
INFO:bodaqs_analysis.io_logger:Logger CSV column matched sidecar: csv_column='timestamp' sidecar_column_id=timestamp dataframe_column=timestamp class=time
INFO:bodaqs

Accordion(children=(VBox(children=(Text(value='event schema\\event_schema - Basic.yaml', description='Schema p…

In [3]:
cfg = controls.get_config()

# If you want hard-stop validation:
errors, warnings = controls.validate(print_to_output=False)
if errors:
    raise ValueError("Fix UI errors before running:\n - " + "\n - ".join(errors))

SCHEMA_PATH = cfg["schema_path"]
PROMPT_FOR_DESCRIPTIONS = cfg["prompt_for_descriptions"]

# ---- Step 2: Validate ranges (bike profile preferred; UI ranges are legacy fallback) ----
SELECTED_DISP_COLS = list(cfg.get("normalize_ranges", {}).keys()) or disp_cols_all
NORMALIZE_RANGES = BIKE_PROFILE_NORMALIZE_RANGES if BIKE_PROFILE is not None else cfg["normalize_ranges"]
ZEROING_ENABLED = cfg["zeroing_enabled"]

missing_ranges = [c for c in SELECTED_DISP_COLS if c not in NORMALIZE_RANGES]
if missing_ranges:
    raise ValueError(
        "Normalization ranges are missing displacement signals:\n  - " + "\n  - ".join(missing_ranges)
    )

# ---- Artifact init (one run_id per batch) ----
store = ArtifactStore(Path("artifacts"))
run_id_base = make_run_id(tz_label="AWST")
run_id = run_id_base
suffix = 1
while store.run_dir(run_id).exists():
    run_id = f"{run_id_base}_{suffix:02d}"
    suffix += 1
ensure_run_is_new(store, run_id=run_id, force=False)
logging.info("Artifact run_id = %s", run_id)

events_all = []
metrics_all = []
sessions_by_id = {}
schema = None
session_ids_in_run = []

for p in CSV_FILES:
    logging.info("Processing %s ...", p.name)

    results = run_macro(
        str(p),
        SCHEMA_PATH,
        zeroing_enabled=cfg["zeroing_enabled"],
        zero_window_s=cfg["zero_window_s"],
        zero_min_samples=cfg["zero_min_samples"],
        clip_0_1=cfg["clip_0_1"],
        active_signal_disp_col=cfg["active_signal_disp_col"],
        active_signal_vel_col=None,
        active_disp_thresh=cfg["active_disp_thresh"],
        active_vel_thresh=cfg["active_vel_thresh"],
        active_window=cfg["active_window"],
        active_padding=cfg["active_padding"],
        active_min_seg=cfg["active_min_seg"],
        normalize_ranges=(None if BIKE_PROFILE is not None else cfg["normalize_ranges"]),
        bike_profile_path=(str(BIKE_PROFILE_PATH) if BIKE_PROFILE is not None else None),
        generic_log_metadata_paths=GENERIC_LOG_METADATA_PATHS,
        fit_import=cfg.get("fit_import"),
        butterworth_smoothing=cfg["butterworth_smoothing"],
        butterworth_generate_residuals=cfg["butterworth_generate_residuals"],
        strict=cfg["strict"],
    )

    session = results["session"]
    sid = str(session["session_id"])

    ensure_session_is_new(store, run_id=run_id, session_id=sid, force=False)
    sessions_by_id[sid] = session
    session_ids_in_run.append(sid)

    source_sha256 = copy_raw_csv_to_source(store=store, run_id=run_id, session_id=sid, csv_path=p)
    aux_sources = copy_session_aux_sources(
        store=store,
        run_id=run_id,
        session_id=sid,
        aux_sources=session.get("source", {}).get("aux_sources"),
    )

    save_session_artifacts(
        store,
        run_id=run_id,
        session_id=sid,
        session_df=session["df"],
        session_meta=session["meta"],
        secondary_stream_dfs=session.get("stream_dfs"),
        secondary_stream_meta=session.get("meta", {}).get("secondary_streams"),
    )
    
    write_session_manifest(
        store,
        run_id=run_id,
        session_id=sid,
        contracts={"session": "v0.x", "events": "v0.x", "metrics": "v0.x"},
        source={"path": "source/input.csv", "sha256": source_sha256},
        aux_sources=aux_sources,
        summary={"n_rows": int(len(session["df"]))},
    )

    if schema is None:
        schema = results.get("schema", None)

    ev = results.get("events")
    mt = results.get("metrics")

    if isinstance(ev, pd.DataFrame) and not ev.empty:
        events_all.append(ev)
        write_events_partitioned_by_schema_id(
            store=store,
            run_id=run_id,
            session_id=sid,
            events_df=ev,
            schema_path=SCHEMA_PATH,
        )

    if isinstance(mt, pd.DataFrame) and not mt.empty:
        metrics_all.append(mt)
        write_metrics_partitioned_by_schema_id(
            store=store,
            run_id=run_id,
            session_id=sid,
            metrics_df=mt,
        )

events_df  = pd.concat(events_all, ignore_index=True) if events_all else pd.DataFrame()
metrics_df = pd.concat(metrics_all, ignore_index=True) if metrics_all else pd.DataFrame()

write_run_manifest(
    store,
    run_id=run_id,
    session_ids=session_ids_in_run,
    timezone_label="AWST",
    pipeline_config={
        "schema_path": str(SCHEMA_PATH),
        "zeroing_enabled": bool(cfg["zeroing_enabled"]),
        "bike_profile_path": str(BIKE_PROFILE_PATH) if BIKE_PROFILE is not None else None,
        "bike_profile_id": BIKE_PROFILE.get("bike_profile_id") if BIKE_PROFILE is not None else None,
        "normalization_ranges_source": "bike_profile" if BIKE_PROFILE is not None else "legacy_ui_ranges",
        "butterworth_smoothing": bool(cfg["butterworth_smoothing"]),
        "butterworth_generate_residuals": bool(cfg["butterworth_generate_residuals"]),
        "ingestion_mode": "strict" if cfg["strict"] else "tolerant",
        "n_files": int(len(CSV_FILES)),
    },
)

logging.info("Wrote artifacts for run_id=%s with %d sessions", run_id, len(session_ids_in_run))
# ---- Optional: prompt for run/session descriptions ----
# Prompting is enabled by default when PROMPT_FOR_DESCRIPTIONS is true.
# For automation/headless runs, set BODAQS_SKIP_DESCRIPTION_PROMPTS=1 (or true/yes).
import os
SKIP_DESCRIPTION_PROMPTS = str(os.getenv("BODAQS_SKIP_DESCRIPTION_PROMPTS", "")).strip().lower() in {"1", "true", "yes", "y"}

if PROMPT_FOR_DESCRIPTIONS and not SKIP_DESCRIPTION_PROMPTS:
    from bodaqs_analysis.artifacts import set_run_description, set_session_description

    run_desc = input(f"Run description for {run_id} (blank to skip): ").strip()
    set_run_description(store, run_id=run_id, description=run_desc)

    for sid in session_ids_in_run:
        s_desc = input(f"Session {sid} description (blank to skip, '.' to stop): ").strip()
        if s_desc == ".":
            break
        set_session_description(store, run_id=run_id, session_id=sid, description=s_desc)
elif PROMPT_FOR_DESCRIPTIONS and SKIP_DESCRIPTION_PROMPTS:
    logging.info("Skipping description prompts (BODAQS_SKIP_DESCRIPTION_PROMPTS is set).")


INFO:root:Artifact run_id = run_2026-04-25T20-31-28_AWST
INFO:root:Processing 2025-10-19_11-42-53_tombstone.CSV ...
INFO:bodaqs_analysis.io_logger:Logger same-stem sidecar not found: expected=C:\Users\Ben Connor\OneDrive\Documents\logs\2025-10-19_11-42-53_tombstone.json
INFO:bodaqs_analysis.io_logger:Logger generic sidecar search path(s): ['config\\sidecar_examples\\current_logger_config_hr_timestamp_sidecar2.json']
INFO:bodaqs_analysis.io_logger:Logger generic sidecar found: config\sidecar_examples\current_logger_config_hr_timestamp_sidecar2.json
INFO:bodaqs_analysis.io_logger:Logger sidecar loaded: path=config\sidecar_examples\current_logger_config_hr_timestamp_sidecar2.json sidecar_kind=generic selected_as_generic=True
INFO:bodaqs_analysis.io_logger:Logger sidecar optional column missing: sidecar_column_id=sample_id csv_ref={'by': 'header', 'header': 'sample_id'}
INFO:bodaqs_analysis.io_logger:Logger CSV column matched sidecar: csv_column='timestamp' sidecar_column_id=timestamp data

Run description for run_2026-04-25T20-31-28_AWST (blank to skip):  
Session 2025-10-19_11-42-53_tombstone description (blank to skip, '.' to stop):  
